# Component I: RNN / LSTM Based Sequential Data Generation

# Sequential Data Generation using LSTM

## (Import Libraries)

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

## Load Dataset

In [2]:
data = [
    "artificial intelligence systems learn patterns from data",
    "sequence models process information step by step",
    "recurrent neural networks are useful for sequence prediction",
    "lstm networks handle long term dependencies",
    "deep learning models improve sequence learning",
    "generative models create new samples from learned patterns",
    "language models predict the next word in a sentence",
    "sequence generation is used in chatbots and assistants",
    "machine learning helps computers learn automatically",
    "training data improves model accuracy",
    "neural networks simulate human brain structures",
    "optimization algorithms improve learning efficiency",
    "technology is transforming modern education",
    "online learning platforms use artificial intelligence",
    "students benefit from intelligent tutoring systems",
    "automation improves productivity and decision making"
]

## Tokenization

In [3]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(data)

total_words = len(tokenizer.word_index) + 1

## Create Input-Output Sequences

In [4]:
input_sequences = []

for line in data:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

max_seq_len = max(len(seq) for seq in input_sequences)

input_sequences = np.array(pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre'))

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

y = tf.keras.utils.to_categorical(y, num_classes=total_words)

## Build LSTM Model

In [ ]:
model = Sequential()
model.add(Embedding(total_words, 64, input_length=max_seq_len-1))
model.add(LSTM(100))
model.add(Dense(total_words, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

## Train Model

In [ ]:
model.fit(X, y, epochs=150, verbose=1)

## Generate Text

In [ ]:
def generate_text(seed_text, next_words):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len-1, padding='pre')

        predicted = np.argmax(model.predict(token_list), axis=-1)[0]

        for word, index in tokenizer.word_index.items():
            if index == predicted:
                seed_text += " " + word
                break

    return seed_text

print(generate_text("machine learning", 5))

# Transformer-Based Sequence Generation

In [16]:
from tensorflow.keras.layers import MultiHeadAttention, LayerNormalization, Dropout
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Lambda

## Positional Encoding Function

In [17]:
def positional_encoding(position, d_model):
    angles = np.arange(position)[:, np.newaxis] / np.power(10000, (2 * (np.arange(d_model)[np.newaxis, :] // 2)) / np.float32(d_model))
    angles[:, 0::2] = np.sin(angles[:, 0::2])
    angles[:, 1::2] = np.cos(angles[:, 1::2])
    return tf.cast(angles[np.newaxis, ...], dtype=tf.float32)

## Build Transformer Block

In [18]:
def transformer_block(inputs, head_size, num_heads, ff_dim, dropout=0):
    x = MultiHeadAttention(num_heads=num_heads, key_dim=head_size)(inputs, inputs)
    x = Dropout(dropout)(x)
    x = LayerNormalization(epsilon=1e-6)(x + inputs)

    ff = Dense(ff_dim, activation="relu")(x)
    ff = Dense(inputs.shape[-1])(ff)

    x = LayerNormalization(epsilon=1e-6)(x + ff)
    return x

## Build Transformer Model

In [ ]:

embedding_dim = 64

inputs = Input(shape=(max_seq_len-1,))
x = Embedding(total_words, embedding_dim)(inputs)

pos_encoding = positional_encoding(max_seq_len-1, embedding_dim)
x = x + pos_encoding[:, :max_seq_len-1, :]

x = transformer_block(x, head_size=64, num_heads=2, ff_dim=128)

x = Lambda(lambda t: t[:, -1, :])(x)

x = Dense(128, activation='relu')(x)
outputs = Dense(total_words, activation='softmax')(x)

model_tf = Model(inputs, outputs)

model_tf.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model_tf.summary()

## Train Transformer Model

In [ ]:
model_tf.fit(X, y, epochs=100, verbose=1)

## Generate Text using Transformer

In [ ]:
def generate_text_transformer(seed_text, next_words):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_seq_len-1, padding='pre')

        predicted = np.argmax(model_tf.predict(token_list), axis=-1)[0]

        for word, index in tokenizer.word_index.items():
            if index == predicted:
                seed_text += " " + word
                break

    return seed_text

print(generate_text_transformer("deep learning", 5))

In [ ]:
lstm_output = generate_text("machine learning", 5)
transformer_output = generate_text_transformer("deep learning", 5)

print("LSTM Output:", lstm_output)
print("Transformer Output:", transformer_output)

with open("generated_sequences.txt", "w") as f:
    f.write("LSTM Generated Text:\n")
    f.write(lstm_output + "\n\n")

    f.write("Transformer Generated Text:\n")
    f.write(transformer_output)

print("\nSaved successfully in generated_sequences.txt")